In [1]:
#We are calling TensorFlow, Google's deep learning library.
import tensorflow as tf
#We are calling the layers and model structures required to build the artificial neural network architecture.
from tensorflow.keras import layers,models
#We call the early stop function to prevent the model from learning and memorizing too much.
from tensorflow.keras.callbacks import EarlyStopping
#We are defining the Fashion MNIST dataset, which comes pre-installed in Keras.
data=tf.keras.datasets.fashion_mnist

In [2]:
#We download the dataset and split it into two parts: Training and Test
(training_images,training_labels),(test_images,test_labels)=data.load_data()

In [3]:
#PREPARING THE DATA FOR CNN
#CNN layers expect the image to be in 4 dimensions (number of images, height, width, number of channels).
#The '1' at the end indicates that the images are black and white. If they were in color, we would write 3.
training_images=training_images.reshape(60000,28,28,1)
test_images=test_images.reshape(10000,28,28,1)
#Pixel values range from 0 to 255. By dividing these values by 255, we compress the pixel values into a range of 0-1. 
#This allows the model to learn faster and more stably.
training_images=training_images/255.0     
test_images=test_images/255.0 

In [4]:
#We are creating a sequential model where the layers will follow each other in order.
model=tf.keras.models.Sequential([
    #It accepts inputs with an input layer of 28x28 pixels and 1 color channel.
    layers.Input(shape=(28,28,1)),
    #Convolution layer: Capture features like edges and corners with 64 different 3x3 filters.
    layers.Conv2D(64,(3,3),activation='relu'),
    #Pooling area: Reduce the size of the image by taking the largest values in the 2x2 areas and highlighting the important details.
    layers.MaxPooling2D(2,2),
    #Second Convolutional Layer: Combining captured features to identify more complex shapes.
    layers.Conv2D(64,(3,3),activation='relu'),
    #Second Pooling Layer: Distilling key features by further reducing the image size.
    layers.MaxPooling2D(2,2),
    #Smoothing Layer: Converts matrix data into a one-dimensional flat line that a classical neural network can read.
    layers.Flatten(),
    #Hidden Layer: Try to correlate features using 128 neurons.
    layers.Dense(128,activation='relu'),
    #Output Layer: Generates probability values for 10 different clothing categories.
    layers.Dense(10,activation='softmax')
])

In [5]:
#We define how the model will be trained.
model.compile(
    optimizer='adam',#An algorithm that updates weights to reduce errors.
    loss='sparse_categorical_crossentropy',#A penalty point system that measures model error.
    metrics=['accuracy'])#A scoreboard that tracks the success rate throughout the training.

In [6]:
#A security system was established to prevent the model from memorizing data.
early_stop=EarlyStopping(
    monitor='val_loss',#What should we follow? (Loss of validation is best.)
    patience=3,#How many epochs of no improvement should we stop the process?
    restore_best_weights=True#Let's go back to the best weights when it stops.
)

In [7]:
#This class acts like an emergency brake, halting the model's training before the predetermined number of epochs are reached,
#the moment the model's training success rate exceeds 95%.
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self,epoch,logs={}):
        if(logs.get('accuracy')>0.95):
            print("\nReached 95% accuracy so cancelling training!")
            self.model.stop_training=True

In [8]:
#We will begin traning the model.
model.fit(
    training_images,  #The visual dataset the model will learn from
    training_labels,  #The visual dataset on which the model will be tested.
    epochs=50,   #Number of training cycles for the model
    validation_split=0.2,  #By allocating twenty percent of the training data, the system tests whether the model has memorized anything at the end of each term.
    callbacks=[early_stop], #It connects the automatic early stop system to the model.
    batch_size=32,  #To avoid overloading the computer's memory, the data is fed into the model in groups of 32 at each step.
    verbose=1  #The system displays the progress bar, error rates, and success percentages live on screen for each term.
)

Epoch 1/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - accuracy: 0.8269 - loss: 0.4677 - val_accuracy: 0.8763 - val_loss: 0.3413
Epoch 2/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - accuracy: 0.8861 - loss: 0.3114 - val_accuracy: 0.8892 - val_loss: 0.3074
Epoch 3/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - accuracy: 0.9029 - loss: 0.2635 - val_accuracy: 0.8987 - val_loss: 0.2742
Epoch 4/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - accuracy: 0.9143 - loss: 0.2309 - val_accuracy: 0.9007 - val_loss: 0.2693
Epoch 5/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - accuracy: 0.9242 - loss: 0.2038 - val_accuracy: 0.9081 - val_loss: 0.2572
Epoch 6/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - accuracy: 0.9321 - loss: 0.1793 - val_accuracy: 0.9095 - val_loss: 0.2580
Epoch 7/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - accuracy: 0.9410 - loss: 0.1573 - val_accuracy: 0.9104 - val_loss: 0.2611
Epoch 8/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - accuracy: 0.9472 - 

In [9]:
#We test the fully trained model on data it has never encountered before.
model.evaluate(test_images,test_labels)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9060 - loss: 0.2672


[0.26722508668899536, 0.906000018119812]

In [10]:
#By looking at the model test images, it can predict which category they belong to.
classifications=model.predict(test_images)
#For the first test image, list the probabilistic values ​​of the 10 different classes generated by the model.
print(classifications[0])
#To check the model's prediction, we print out what that initial test image actually was.    
print(test_labels[0])

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
[4.7521759e-08 1.0823512e-07 1.7670347e-08 8.4817430e-08 8.1359482e-11
 2.9881898e-04 1.7204906e-08 4.0606705e-05 2.0663427e-07 9.9966013e-01]
9


In [11]:
model.summary()#We are printing the model summary.

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 26, 26, 64)          │             640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 13, 13, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 11, 11, 64)          │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 5, 5, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 1600)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         204,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 731,360 (2.79 MB)

 Trainable params: 243,786 (952.29 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 487,574 (1.86 MB)